# Custom dataset collation

In this notebook we create a custom collation function for the `HyraxRandomDataset`.

**What is collation?** When a DataLoader assembles individual samples into a batch, it
calls a *collate* function to combine them. Hyrax provides a default implementation, but
custom collation is required when samples have non-uniform shapes. A common example is
light-curve data: different objects may have different numbers of photometric observations,
so shorter sequences must be padded to a common length and a Boolean *mask* must indicate
which values are padding. The model then uses the mask to ignore padded positions.

For simplicity we use a dataset with a **uniform shape**, so we can focus on the
mechanics of creating a custom collate function rather than the padding logic. A mask
is still created, but all its values will be `False` (no padding needed).

We begin by creating a `Hyrax` instance.

In [ ]:
from hyrax import Hyrax

h = Hyrax()

Next we define a data request. The data request is a nested dictionary that tells
Hyrax which datasets to load and which fields to expose. The top-level key `"train"`
names the data split; inside it, `"data"` is the *friendly name* assigned to this
particular dataset. The friendly name appears as a key in every sample returned by the
DataProvider, so choosing a descriptive name (e.g. `"spectra"` or `"images"`) helps
keep multi-dataset workflows readable.

In [ ]:
data_request = {
    "train": {
        "data": {
            "dataset_class": "HyraxRandomDataset",
            "data_location": "./data",
            "fields": ["object_id", "image", "label"],
            "split_fraction": 1.0,
            "primary_id_field": "object_id",
        },
    }
}

h.set_config("data_request", data_request)

## Add a custom collation function

A custom `collate` static method can be attached directly to the dataset class.
Hyrax detects it automatically when the dataset is prepared and uses it in place
of the default collation logic.

The function receives a list of per-sample field dictionaries for the dataset's
friendly name — each element corresponds to one item from `dataset[i]["data"]`.
It should return a single dictionary of collated arrays.

**Important: guard against missing fields.** A dataset class may expose more
fields than a user requests in their `data_request`. The collate function is
always called when it exists, even if only a subset of fields was requested.
To avoid `KeyError` exceptions when a field is absent, **wrap each field’s
collation in an `if` check** on the first sample (e.g.
`if "image" in samples[0]`). Without these guards the collate function will
fail whenever a user omits a field from their request.

In production, the function should be a method on the dataset class. Attaching it
dynamically here, as shown below, is convenient for prototyping in a notebook.

In [ ]:
from hyrax.datasets.random.hyrax_random_dataset import HyraxRandomDataset
import numpy as np


@staticmethod
def collate(samples: list[dict]) -> dict:
    """Collate a list of dictionaries into a single batch.

    This method takes a list of samples and collates them into a single batch.
    The returned batch dictionary will contain the following keys (when present
    in the requested fields):

    - ``object_id``: Numpy array of object IDs for the samples in the batch.
    - ``image``: Numpy array of stacked images for the samples in the batch.
    - ``mask``: Numpy array of masks with the same shape as the images. Derived
      from ``image`` and only present when ``image`` is requested.
    - ``label``: Numpy array of labels for the samples in the batch.

    Each field is guarded with an ``if`` check so that the function works
    correctly even when the user has not requested every field the dataset
    exposes.

    Parameters
    ----------
    samples : list of dict
        A list of samples to collate.

    Returns
    -------
    dict
        A single dictionary containing the collated data as numpy arrays.
    """
    collated_data = {}

    if "object_id" in samples[0]:
        collated_data["object_id"] = np.array([sample["object_id"] for sample in samples])

    if "image" in samples[0]:
        # vertically stack all the images into a single NumPy array
        collated_data["image"] = np.stack([sample["image"] for sample in samples], axis=0)
        # create a "mask" that has the same shape as the "image" stack.
        collated_data["mask"] = np.zeros_like(collated_data["image"], dtype=bool)

    if "label" in samples[0]:
        collated_data["label"] = np.array([sample["label"] for sample in samples])

    return collated_data


# Attach the collation function to the HyraxRandomDataset class
HyraxRandomDataset.collate = collate

## Custom field-level collate functions

Instead of defining a `collate` function for the entire dataset, it is possible to
isolate the collation logic for each field and create `collate_*` functions.
This has the benefit of not requiring `if` statement guards since Hyrax will
automatically pull the relevant `collate_*` functions for requested fields.
The `*` must be replaced with the name of the field to be collated (see example below).

**Important: Incompatible with dataset-level `collate` function.** Hyrax expects
either field-level collate functions **OR** a dataset-level collate function.
Defining both will cause Hyrax to throw an error when training is attempted.

Hyrax defines default behavior for fields which do not have a collate function defined.
The default behavior simply attempts to stack the given data into a single `numpy` array,
without adding padding or masking. This works for scalar data or uniformly shaped data,
which is why the `object_id` and `label` fields above do not have a custom `collate_*`
function defined. Desired results may not be achieved in other scenarios and defining
custom collate functions allows to explicitly handle complex or non-uniform data fields.

As with the dataset-level `collate` function, in production the field-level collate
functions should be methods on the dataset class.

In [ ]:
@staticmethod
def collate_image(samples: list[dict]) -> dict:
    """Collate a list of dictionaries into a single batch.

    This method takes a list of samples and collates the the values for ``image`` into a single batch.
    The returned batch dictionary will contain the following keys (when present
    in the requested fields):

    - ``image``: Numpy array of stacked images for the samples in the batch.
    - ``mask``: Numpy array of masks with the same shape as the images. Derived
      from ``image`` and only present when ``image`` is requested.

    Parameters
    ----------
    samples : list of dict
        A list of samples to collate.

    Returns
    -------
    dict
        A single dictionary containing the collated data as numpy arrays.
    """
    collated_data = {}

    # vertically stack all the images into a single NumPy array
    collated_data["image"] = np.stack([sample["image"] for sample in samples], axis=0)
    # create a "mask" that has the same shape as the "image" stack.
    collated_data["mask"] = np.zeros_like(collated_data["image"], dtype=bool)

    return collated_data


# Attach the collation function to the HyraxRandomDataset class
# The ``object_id`` and ``label`` fields will use Hyrax's default field collation
HyraxRandomDataset.collate_image = collate_image

## Prepare the dataset

`h.prepare()` instantiates the requested dataset classes and returns a dictionary
keyed by split name. It also picks up the custom `collate` method we just attached
to `HyraxRandomDataset`. Hyrax will error if it detects that both a
dataset-level `collate` function and a field-level collation function are defined,
so we delete the dataset-level collate function first.

In [ ]:
# Remove the "collate" function to avoid raising an error
delattr(HyraxRandomDataset, "collate")
dataset = h.prepare()

# Access the "train" data group for clarity in the next steps
train_dataset = dataset["train"]

## Inspect individual samples

We request two samples by index and print their field types and shapes. Each sample
is a dictionary whose keys are the friendly name (`"data"`) plus a top-level
`"object_id"` entry. The fields we requested live under `sample["data"]`.

In [ ]:
sample_0 = train_dataset[0]
sample_1 = train_dataset[1]

In [ ]:
print("Type and shape / value of fields in sample 0:")
# Fields are nested under the friendly name "data"
print(f"object_id type: {type(sample_0['data']['object_id'])}, value: {sample_0['data']['object_id']}")
print(f"image type: {type(sample_0['data']['image'])}, shape: {sample_0['data']['image'].shape}")
print(f"label type: {type(sample_0['data']['label'])}, value: {sample_0['data']['label']}")

print("\nSample 0")
print(sample_0)

## Collate two samples manually

We call `train_dataset.collate()` directly to see the result. During training, the
PyTorch DataLoader calls this same method automatically to build each mini-batch.

In [ ]:
collated = train_dataset.collate([sample_0, sample_1])

In [ ]:
print("Fields in collated output:", collated["data"].keys())
print("Collated types and shapes from samples 0 and 1:")
# Results are nested under the friendly name "data", mirroring the sample structure
print(f"image type: {type(collated['data']['image'])}, shape: {collated['data']['image'].shape}")
print(f"object_id type: {type(collated['data']['object_id'])}, shape: {collated['data']['object_id'].shape}")
print(f"label type: {type(collated['data']['label'])}, shape: {collated['data']['label'].shape}")
print(f"mask type: {type(collated['data']['mask'])}, shape: {collated['data']['mask'].shape}")

The batch now contains a `mask` field with the same type and shape as `image`. All
values are `False` here because the data was uniformly shaped and required no padding.
In a real use case, positions that were padded would be `True`, signalling to the model
that those values should be ignored.

When `HyraxRandomDataset` is used in a training run, the DataLoader will call this
same `collate` method automatically — no additional configuration is required.

We can also see that if the `collate_image` function is removed, then Hyrax uses the
default collation for every field, including `image`. This leads to the same collated output
output with the exception of there being no `mask` field.

In [ ]:
delattr(HyraxRandomDataset, "collate_image")
dataset = h.prepare()

# Access the "train" data group for clarity in the next steps
train_dataset = dataset["train"]
collated = train_dataset.collate([sample_0, sample_1])

In [ ]:
print("Fields in collated output:", collated["data"].keys())
print("Collated types and shapes from samples 0 and 1:")
# Results are nested under the friendly name "data", mirroring the sample structure
print(f"image type: {type(collated['data']['image'])}, shape: {collated['data']['image'].shape}")
print(f"object_id type: {type(collated['data']['object_id'])}, shape: {collated['data']['object_id'].shape}")
print(f"label type: {type(collated['data']['label'])}, shape: {collated['data']['label'].shape}")